In [1]:
DATA_PATH = "data/Train"

# Data 

In [2]:
from torch.utils.data import Dataset
import glob 
import cv2 
import torchvision.transforms as transforms
import os 


class SegmentationDataset(Dataset): 
    def __init__(self, data_dir, transform=None): 

        self.data_dir = data_dir 
        self.transform = transform

        self.images = sorted(glob.glob(os.path.join(data_dir, "Image", "*.jpg")))
        self.masks = sorted(glob.glob(os.path.join(data_dir, "Mask", "*.png")))


    def __len__(self): 
        return len(self.images)


    def __getitem__(self, idx): 

        image_path = self.images[idx]
        mask_path = self.masks[idx]

        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # Chuyển BGR sang RGB
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)


        if self.transform:
            aug = self.transform(image=image, mask=mask)
            image = aug['image']
            mask = aug['mask']

        image = transforms.ToTensor()(image)
        mask = transforms.ToTensor()(mask)


        return image, mask

In [3]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
import os 
from torch.utils.data import DataLoader
import lightning as pl

/home/thviet/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# data module 
from torch.utils.data import random_split



class SegmentationDataModule(pl.LightningDataModule):
    def __init__(self, data_dir, batch_size=8, num_workers=4, image_size=(256, 256)):
        super().__init__()
        self.data_dir = data_dir
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.image_size = image_size
        
    def setup(self, stage=None):

        self.train_transform = A.Compose([
            A.Resize(height=self.image_size[0], width=self.image_size[1]),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.RandomBrightnessContrast(p=0.2),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ])
        
        self.val_transform = A.Compose([
            A.Resize(height=self.image_size[0], width=self.image_size[1]),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ])
        
        # Tạo dataset
        self.train_dataset = SegmentationDataset(
            DATA_PATH,
            transform=self.train_transform
        )
        
        self.train_dataset, self.val_dataset = random_split(
            self.train_dataset, 
            [len(self.train_dataset) - int(0.2 * len(self.train_dataset)), int(0.2 * len(self.train_dataset))]
        )

        
    def train_dataloader(self):
        return DataLoader(
            self.train_dataset, 
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
            pin_memory=True
        )
    
    def val_dataloader(self):
        return DataLoader(
            self.val_dataset, 
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=True
        )


# Model 

## FCN 

In [5]:
import torch.nn as nn 
from torchvision.models import vgg16

class FCN(nn.Module):
    def __init__(self, n_class):
        super().__init__()
        self.n_class = n_class
        

        vgg_model = vgg16(pretrained=True)
        self.vgg_features = vgg_model.features
        
        self.fcn = nn.Sequential(
            nn.Conv2d(512, 4096, kernel_size=7, padding=3),
            nn.ReLU(inplace=True),
            nn.Dropout2d(),
            nn.Conv2d(4096, 4096, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Dropout2d(),
            nn.Conv2d(4096, self.n_class, kernel_size=1)
        )
        
        self.upsample32 = nn.ConvTranspose2d(
            self.n_class, self.n_class, kernel_size=64, stride=32, padding=16
        )
        
    def forward(self, x):
        
        input_size = x.size()[2:]
        features = self.vgg_features(x)
        
        x = self.fcn(features)
        
        x = self.upsample32(x)
        x = x[:, :, :input_size[0], :input_size[1]]
        
        return x

## Unet 


In [6]:
import torch 

class UNetDoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        return self.double_conv(x)

class UNet(nn.Module):
    def __init__(self, n_channels=3, n_classes=1):
        super().__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes
        
        # Encoder
        self.inc = UNetDoubleConv(n_channels, 64)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), UNetDoubleConv(64, 128))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), UNetDoubleConv(128, 256))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), UNetDoubleConv(256, 512))
        self.down4 = nn.Sequential(nn.MaxPool2d(2), UNetDoubleConv(512, 1024))
        
        # Decoder
        self.up1 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.up_conv1 = UNetDoubleConv(1024, 512)
        
        self.up2 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.up_conv2 = UNetDoubleConv(512, 256)
        
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.up_conv3 = UNetDoubleConv(256, 128)
        
        self.up4 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.up_conv4 = UNetDoubleConv(128, 64)
        
        self.outc = nn.Conv2d(64, n_classes, kernel_size=1)
    
    def forward(self, x):
        # Encoder
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        
        # Decoder với skip connections
        x = self.up1(x5)
        x = torch.cat([x4, x], dim=1)
        x = self.up_conv1(x)
        
        x = self.up2(x)
        x = torch.cat([x3, x], dim=1)
        x = self.up_conv2(x)
        
        x = self.up3(x)
        x = torch.cat([x2, x], dim=1)
        x = self.up_conv3(x)
        
        x = self.up4(x)
        x = torch.cat([x1, x], dim=1)
        x = self.up_conv4(x)
        
        # Output layer
        x = self.outc(x)
        return x

# Lightning module 

In [17]:
class SegmentationModel(pl.LightningModule):
    def __init__(self, model_name, n_classes, learning_rate=1e-4):
        super().__init__()
        self.model_name = model_name
        self.n_classes = n_classes
        self.learning_rate = learning_rate
    
        if model_name == "fcn":
            self.net = FCN(n_classes)
        elif model_name == "unet":
            self.net = UNet(n_channels=3, n_classes=n_classes)
        else:
            raise ValueError(f"Model {model_name} không được hỗ trợ.")
        
        self.criterion = nn.CrossEntropyLoss() if n_classes > 1 else nn.BCEWithLogitsLoss()
        
        self.save_hyperparameters()
    
    def forward(self, x):
        return self.net(x)
    
    def shared_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        
        if self.n_classes == 1:
            y = y.float() 
        else:
            y = y.squeeze(1).long()  
            
        # Tính loss
        loss = self.criterion(y_hat, y)
        return loss, y_hat, y
    
    def training_step(self, batch, batch_idx):
        loss, preds, targets = self.shared_step(batch, batch_idx)
        self.log('train_loss', loss, prog_bar=True)
        return loss
    
    def validation_step(self, batch, batch_idx):
        loss, preds, targets = self.shared_step(batch, batch_idx)
        self.log('val_loss', loss, prog_bar=True)
        
        # Tính IoU nếu là multi-class segmentation
        if self.n_classes > 1:
            preds = torch.argmax(preds, dim=1)
            iou = self.calculate_iou(preds, targets)
            self.log('val_iou', iou, prog_bar=True)
        
        return loss
    
    def calculate_iou(self, preds, targets):
        # IoU calculation for validation
        if targets.dim() == 4:  # Nếu targets có shape [B, 1, H, W]
            targets = targets.squeeze(1)  # -> [B, H, W]


        intersection = torch.logical_and(targets, preds).sum((1, 2)).float()
        union = torch.logical_or(targets, preds).sum((1, 2)).float()
        
        iou = (intersection + 1e-8) / (union + 1e-8)
        return iou.mean()
    
    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.learning_rate)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, 
            mode='min', 
            factor=0.5, 
            patience=5
        )
        
        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'monitor': 'val_loss',
            }
        }

# Training 

In [18]:
from lightning import Trainer
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping
import wandb 
from lightning.pytorch.loggers import WandbLogger
from dotenv import load_dotenv 

load_dotenv('.env')

False

In [19]:
def train_segmentation_model(data_dir, model_name, n_classes, batch_size=8, max_epochs=50):

    wandb.login(
        key = os.getenv("WANDB_KEY_API", "your_wandb_api_key_here")  # Replace with your actual API key
    )

    # Setup data module
    data_module = SegmentationDataModule(
        data_dir=data_dir,
        batch_size=batch_size,
        num_workers=4,
        image_size=(256, 256)
    )
    
    # Setup model
    model = SegmentationModel(
        model_name=model_name,
        n_classes=n_classes,
        learning_rate=1e-4
    )
    
    # Setup callbacks
    checkpoint_callback = ModelCheckpoint(
        monitor='val_loss',
        dirpath=f'./checkpoints/{model_name}',
        filename=f'{model_name}'+'-{epoch:02d}-{val_loss:.4f}',
        save_top_k=3,
        mode='min'
    )
    
    early_stop_callback = EarlyStopping(
        monitor='val_loss',
        patience=10,
        mode='min'
    )

    logger = WandbLogger(
        project='segmentation_project',
        name=f'{model_name}_experiment',
        log_model=True
    )
    

    trainer = Trainer(
        max_epochs=max_epochs,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        devices=1,
        callbacks=[checkpoint_callback, early_stop_callback], 
        logger=logger
    )
    
    trainer.fit(model, data_module)
    
    return model, trainer

In [20]:
import numpy as np 
import matplotlib.pyplot as plt


def visualize_predictions(model, dataloader, num_images=3):
    model.eval()
    device = next(model.parameters()).device
    
    fig, axes = plt.subplots(num_images, 3, figsize=(15, 5*num_images))
    
    dataiter = iter(dataloader)
    with torch.no_grad():
        for i in range(num_images):
            images, masks = next(dataiter)
            images = images.to(device)
            
            # Get predictions
            outputs = model(images)
            
            if model.n_classes > 1:
                preds = torch.argmax(outputs, dim=1)
            else:
                preds = (outputs > 0).float()
            
            # Convert to numpy for visualization
            image = images[0].cpu().permute(1, 2, 0).numpy()
            image = (image * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406]))
            image = np.clip(image, 0, 1)
            
            mask = masks[0].cpu().numpy()
            pred = preds[0].cpu().numpy()
            
            # Plot
            axes[i, 0].imshow(image)
            axes[i, 0].set_title("Original Image")
            axes[i, 0].axis('off')
            
            axes[i, 1].imshow(mask, cmap='jet')
            axes[i, 1].set_title("Ground Truth")
            axes[i, 1].axis('off')
            
            axes[i, 2].imshow(pred, cmap='jet')
            axes[i, 2].set_title("Prediction")
            axes[i, 2].axis('off')
    
    plt.tight_layout()
    plt.show()

In [21]:
fcn_model, fcn_trainer = train_segmentation_model(
        data_dir="data/Train",
        model_name="fcn",
        n_classes=2,  
        batch_size=8,
        max_epochs=2
    )



wandb: WARNING Calling wandb.login() after wandb.init() has no effect.
/workspace/thviet/paracrawl/tools/miniconda3/envs/img_env/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/workspace/thviet/paracrawl/tools/miniconda3/envs/img_env/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/workspace/thviet/paracrawl/tools/miniconda3/envs/img_env/lib/python3.11/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your i

Epoch 1: 100%|██████████| 109/109 [00:24<00:00,  4.44it/s, v_num=ggjn, train_loss=0.148, val_loss=0.129, val_iou=0.791] 

`Trainer.fit` stopped: `max_epochs=2` reached.


Epoch 1: 100%|██████████| 109/109 [00:37<00:00,  2.92it/s, v_num=ggjn, train_loss=0.148, val_loss=0.129, val_iou=0.791]


In [ ]:
unet_model, unet_trainer = train_segmentation_model(
        data_dir="data/Train",
        model_name="unet",
        n_classes=2,  
        batch_size=8,
        max_epochs=2
    )
    